The Gold Layer contains business-ready data that can be directly used for reporting, dashboards, KPIs, and decision-making.

In [0]:
%python

from pyspark.sql.functions import *

silver_df = spark.table(
    "retail_lakehouse.silver.superstore_clean"
)

display(silver_df)

In [0]:
%python

gold_df = (
    silver_df
    .withColumn(
        "sales_num",
        col("sales").cast("double")
    )
    .withColumn(
        "profit_num",
        col("profit").cast("double")
    )
    .withColumn(
        "discount_num",
        col("discount").cast("double")
    )
)

gold_df.printSchema()

In [0]:
%python

display(
    silver_df.select(
        "sales",
        "quantity",
        "discount",
        "profit"
    )
)

In [0]:
%python

gold_df = (
    silver_df
    .withColumn(
        "sales_num",
        expr("try_cast(sales as double)")
    )
    .withColumn(
        "profit_num",
        expr("try_cast(profit as double)")
    )
    .withColumn(
        "discount_num",
        expr("try_cast(discount as double)")
    )
    .withColumn(
        "quantity_num",
        expr("try_cast(quantity as int)")
    )
)

In [0]:
%python

gold_df.printSchema()

In [0]:
%python
sales_by_region = (
    gold_df
    .groupBy("region")
    .agg(
        round(
            sum("sales_num"),
            0
        ).alias("total_sales"),

        round(
            sum("profit_num"),
            0
        ).alias("total_profit")
    )
)

display(sales_by_region)

In [0]:
%python

sales_by_region.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.sales_by_region"
)

In [0]:
%python

sales_by_category = (
    gold_df
    .groupBy("category")
    .agg(
        round(
            sum("sales_num"),
            2
        ).alias("total_sales"),

        round(
            sum("profit_num"),
            2
        ).alias("total_profit")
    )
)

display(sales_by_category)

In [0]:
%python

sales_by_category.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.sales_by_category"
)

In [0]:
%python

sales_by_segment = (
    gold_df
    .groupBy("segment")
    .agg(
        round(
            sum("sales_num"),
            2
        ).alias("total_sales"),

        round(
            sum("profit_num"),
            2
        ).alias("total_profit")
    )
)

display(sales_by_segment)

In [0]:
%python

sales_by_segment.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.sales_by_segment"
)

In [0]:
%python

top_products = (
    gold_df
    .groupBy("product_name")
    .agg(
        round(
            sum("sales_num"),
            2
        ).alias("total_sales")
    )
    .orderBy(
        desc("total_sales")
    )
    .limit(10)
)

display(top_products)

In [0]:
%python

top_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.top_products"
)

In [0]:
%python

display(
    spark.table(
        "retail_lakehouse.gold.sales_by_region"
    )
)

In [0]:
%python

display(
    spark.table(
        "retail_lakehouse.gold.sales_by_category"
    )
)

In [0]:
%python

display(
    spark.table(
        "retail_lakehouse.gold.sales_by_segment"
    )
)

In [0]:
%python

display(
    spark.table(
        "retail_lakehouse.gold.top_products"
    )
)

In [0]:
%python

discount_analysis = (
    gold_df
    .groupBy("discount_num")
    .agg(
        round(sum("sales_num"),2).alias("total_sales"),
        round(sum("profit_num"),2).alias("total_profit")
    )
    .orderBy("discount_num")
)

display(discount_analysis)

In [0]:
%python
discount_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.discount_analysis"
)

In [0]:
%python

state_profit = (
    gold_df
    .groupBy("state")
    .agg(
        round(sum("profit_num"),2)
        .alias("total_profit")
    )
    .orderBy(desc("total_profit"))
)

display(state_profit)

In [0]:
%python
state_profit.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.state_profit"
)

In [0]:
%python

profitable_products = (
    gold_df
    .groupBy("product_name")
    .agg(
        round(
            sum("profit_num"),
            2
        ).alias("total_profit")
    )
    .orderBy(
        desc("total_profit")
    )
    .limit(10)
)

display(profitable_products)

In [0]:
%python
profitable_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.profitable_products"
)

In [0]:
%python

loss_products = (
    gold_df
    .groupBy("product_name")
    .agg(
        round(
            sum("profit_num"),
            2
        ).alias("total_profit")
    )
    .orderBy("total_profit")
    .limit(10)
)

display(loss_products)

In [0]:
%python
loss_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.loss_products"
)

In [0]:
%python

customer_sales = (
    gold_df
    .groupBy("customer_name")
    .agg(
        round(
            sum("sales_num"),
            2
        ).alias("total_sales")
    )
    .orderBy(
        desc("total_sales")
    )
)

display(customer_sales)

In [0]:
%python
customer_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "retail_lakehouse.gold.customer_sales"
)